In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from geopy.distance import great_circle
import numpy as np

class Station:
    def __init__(self):
        self.stations = {}

    def build_all_stations(self):
        self.stations.setdefault('Portal Norte', [0, (4.754228, -74.046161)])
        self.stations.setdefault('Toberin', [1, (4.746185, -74.047279)])
        self.stations.setdefault('Calle 161', [2, (4.742706, -74.047863)])
        self.stations.setdefault('Mazuren', [3, (4.734499, -74.049242)])
        self.stations.setdefault('Calle 146', [4, (4.730832, -74.049868)])
        self.stations.setdefault('Calle 142', [5, (4.726947, -74.050305)])
        self.stations.setdefault('Alcala', [6, (4.720287, -74.051641)])
        self.stations.setdefault('Prado', [7, (4.713173, -74.052682)])
        self.stations.setdefault('Calle 127', [8, (4.704787, -74.054230)])
        self.stations.setdefault('Pepe Sierra', [9, (4.698795, -74.055251)])
        self.stations.setdefault('Calle 106', [10, (4.691557, -74.056421)])
        self.stations.setdefault('Calle 100', [11, (4.684800, -74.057570)])
        self.stations.setdefault('La Castellana', [12, (4.676243, -74.063387)])
        self.stations.setdefault('NQS-Calle 75', [13, (4.670653, -74.070593)])
        self.stations.setdefault('AV. Chile', [14, (4.665962, -74.074819)])
        self.stations.setdefault('Simon Bolivar', [15, (4.658008, -74.077797)])
        self.stations.setdefault('Movistar Arena', [16, (4.650119, -74.078363)])
        self.stations.setdefault('Campin - U. Antonio Nariño', [17, (4.644847, -74.078777)])
        self.stations.setdefault('AV. El Dorado', [18, (4.630541, -74.079891)])
        self.stations.setdefault('CAD', [19, (4.622983, -74.084559)])
        self.stations.setdefault('Paloquemao', [20, (4.617084, -74.089525)])
        self.stations.setdefault('Ricaurte', [21, (4.612523, -74.093075)])
        self.stations.setdefault('San Façon', [22, (4.609549, -74.086540)])
        self.stations.setdefault('De La Sabana', [23, (4.605659, -74.082138)])
        self.stations.setdefault('AV. Jiménez', [24, (4.603037, -74.079164)])
        self.stations.setdefault('Virrey', [25, (4.675857, -74.059144)])
        self.stations.setdefault('Calle 85', [26, (4.671851, -74.059702)])
        self.stations.setdefault('Héroes', [27, (4.668311, -74.060210)])
        self.stations.setdefault('Calle 76', [28, (4.664031, -74.061083)])
        self.stations.setdefault('Calle 72', [29, (4.659261, -74.061922)])
        self.stations.setdefault('Flores', [30, (4.654878, -74.063021)])
        self.stations.setdefault('Calle 63', [31, (4.648914, -74.064810)])
        self.stations.setdefault('Calle 57', [32, (4.642917, -74.065879)])
        self.stations.setdefault('Calle 45', [34, (4.632661, -74.067665)])
        self.stations.setdefault('AV. 39', [35, (4.627184, -74.068643)])
        self.stations.setdefault('Calle 34', [36, (4.621390, -74.069805)])
        self.stations.setdefault('Calle 22', [38, (4.611033, -74.075079)])
        self.stations.setdefault('Calle 19', [39, (4.608302, -74.076608)])
        self.stations.setdefault('Concejo de Bogotá', [40, (4.626496, -74.080722)])
        self.stations.setdefault('Centro Memoria', [41, (4.621915, -74.077436)])
        self.stations.setdefault('U. Nacional', [42, (4.636493, -74.079328)])


    def distance(self, frm, to):
        return float(great_circle(frm, to).meters)

    def generate_data_for_supervised_learning(self, graph):
        X = []
        y = []
        for vertex in graph.vert_dict.values():
            origin_coords = vertex.get_coordinates()
            for neighbor in vertex.get_vertex_childrens():
                destination_coords = neighbor.get_coordinates()
                distance = self.distance(origin_coords, destination_coords)
                X.append([origin_coords[0], origin_coords[1], destination_coords[0], destination_coords[1]])
                y.append(distance)
        return np.array(X), np.array(y)

class Vertex:
    def __init__(self, key, coordinates):
        self.id = key
        self.coordinates = coordinates
        self.connected_to = {}

    def add_neighbor(self, neighbor, weight=0):
        self.connected_to[neighbor] = weight

    def get_vertex_childrens(self):
        return self.connected_to.keys()

    def get_coordinates(self):
        return self.coordinates

class Graph:
    def __init__(self):
        self.vert_dict = {}

    def add_vertex(self, key, coordinates):
        new_vertex = Vertex(key, coordinates)
        self.vert_dict[key] = new_vertex
        return new_vertex

    def add_edge(self, frm, to, cost=0):
        if frm not in self.vert_dict:
            self.add_vertex(frm, station.stations[frm][1])
        if to not in self.vert_dict:
            self.add_vertex(to, station.stations[to][1])
        self.vert_dict[frm].add_neighbor(self.vert_dict[to], cost)

# Crear el grafo con los datos de las estaciones
station = Station()
station.build_all_stations()

graph = Graph()

# Crear conexiones
graph.add_edge('Portal Norte', 'Movistar Arena', station.distance(station.stations['Portal Norte'][1], station.stations['Movistar Arena'][1]))
graph.add_edge('Movistar Arena', 'Virrey', station.distance(station.stations['Movistar Arena'][1], station.stations['Virrey'][1]))

# Generar los datos para el aprendizaje supervisado
X, y = station.generate_data_for_supervised_learning(graph)

# División de los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear el modelo de Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Entrenamos el modelo
model.fit(X_train, y_train)

# Realiza predicciones
y_pred = model.predict(X_test)

# Evaluar el modelo
mse = mean_squared_error(y_test, y_pred)
print(f'Error cuadrático medio (Random Forest): {mse}')

# Calcular RMSE para una mejor interpretación
rmse = np.sqrt(mse)
print(f'Raíz del error cuadrático medio (RMSE): {rmse}')


Error cuadrático medio (Random Forest): 73041190.61603697
Raíz del error cuadrático medio (RMSE): 8546.413903856808
